In [ ]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad, quad

In [5]:
"""
Cálculo TOTALMENTE numérico da integral tripla:

I1 = ∫₀⁵ dq ∫₀²π dφ ∫₀^∞ dk  k * q² / [(k²+μ²)(k²-2kq cosφ + q²+μ²)]

Mudança de variável para mapear k ∈ [0,∞) → t ∈ [0,1):
    k = t / (1 - t),  dk = 1/(1-t)²  dt

Tudo feito numericamente com scipy.integrate.tplquad.
"""

import numpy as np
from scipy import integrate
import warnings

# ── Parâmetro ─────────────────────────────────────────────────────────────────
mu = 1.0

# ── Integrando após mudança de variável k = t/(1-t) ──────────────────────────
def integrand(t, phi, q):
    if t >= 1.0:
        return 0.0
    k    = t / (1.0 - t)
    jac  = 1.0 / (1.0 - t)**2          # dk/dt
    num  = k * q**2
    den1 = k**2 + mu**2
    den2 = k**2 - 2*k*q*np.cos(phi) + q**2 + mu**2
    if den1 == 0 or den2 == 0:
        return 0.0
    return (num / (den1 * den2)) * jac

# ── Limites ───────────────────────────────────────────────────────────────────
t_lo,   t_hi   = 0.0, 0.999999   # evita k → ∞ na borda
phi_lo, phi_hi = 0.0, 2*np.pi
q_lo,   q_hi   = 0.0, 5.0

print("=" * 60)
print(f"  Cálculo numérico de I₁  (μ = {mu:.2f})")
print("=" * 60)
print("\n  Integrando com scipy.tplquad ...")
print("  Ordem: t(→k) mais interna, φ no meio, q mais externa\n")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result, err = integrate.tplquad(
        integrand,
        q_lo, q_hi,          # limites de q (mais externo)
        phi_lo, phi_hi,      # limites de φ (meio)
        t_lo, t_hi,          # limites de t → k (mais interno)
        epsabs=1e-8,
        epsrel=1e-8,
    )
    for w in caught:
        print(f"  ⚠ Aviso: {w.message}")

print("=" * 60)
print(f"  I₁    = {result:.8f}")
print(f"  Erro  ≈ {err:.2e}")
print("=" * 60)

  Cálculo numérico de I₁  (μ = 1.00)

  Integrando com scipy.tplquad ...
  Ordem: t(→k) mais interna, φ no meio, q mais externa

  I₁    = 48.63953483
  Erro  ≈ 1.63e-07


In [6]:
"""
Cálculo TOTALMENTE numérico da integral tripla:

I1 = ∫₀⁵ dq ∫₀²π dφ ∫₀^10 dk  k * q² / [(k²+μ²)(k²-2kq cosφ + q²+μ²)]

Sem nenhuma mudança de variável - tudo numérico direto com scipy.tplquad.
"""

import numpy as np
from scipy import integrate
import warnings

# ── Parâmetro ─────────────────────────────────────────────────────────────────
mu = 1.0

# ── Integrando original ───────────────────────────────────────────────────────
def integrand(k, phi, q):
    num  = k * q**2
    den1 = k**2 + mu**2
    den2 = k**2 - 2*k*q*np.cos(phi) + q**2 + mu**2
    if den1 == 0 or den2 == 0:
        return 0.0
    return num / (den1 * den2)

# ── Limites ───────────────────────────────────────────────────────────────────
k_lo,   k_hi   = 0.0, 10.0
phi_lo, phi_hi = 0.0, 2*np.pi
q_lo,   q_hi   = 0.0, 5.0

print("=" * 60)
print(f"  Cálculo numérico de I₁  (μ = {mu:.2f})")
print(f"  k ∈ [0, 10]")
print("=" * 60)
print("\n  Integrando com scipy.tplquad ...")
print("  Ordem: k mais interna, φ no meio, q mais externa\n")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result, err = integrate.tplquad(
        integrand,
        q_lo, q_hi,
        phi_lo, phi_hi,
        k_lo, k_hi,
        epsabs=1e-4,
        epsrel=1e-4,
    )
    for w in caught:
        print(f"  ⚠ Aviso: {w.message}")

print("=" * 60)
print(f"  I₁    = {result:.8f}")
print(f"  Erro  ≈ {err:.2e}")
print("=" * 60)

  Cálculo numérico de I₁  (μ = 1.00)
  k ∈ [0, 10]

  Integrando com scipy.tplquad ...
  Ordem: k mais interna, φ no meio, q mais externa

  I₁    = 47.23593540
  Erro  ≈ 1.72e-03


In [ ]:
# Load experimental data
atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [ ]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'

In [ ]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q2, phi, mg, a1, a2, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q2, phi, mg, a1, a2, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  


def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [ ]:
n_points = 1750

def full_int(mg, a1, a2, m2_func, q2_val, sqrt_s):
    # Garante que q_val seja array 1D
    q2_val = np.atleast_1d(q2_val)
    results = []

    for q2 in q2_val:
        def integrand(y, x, mg, a1, a2, m2_func, q2_val):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s
            result = k * (
                T_1(k, q2_val, phi, mg, a1, a2, m2_func)
                - T_2(k, q2_val, phi, mg, a1, a2, m2_func)
            ) * jacobian
            return result

        def inner_integral(x):
            integral_real = fixed_quad(
                lambda y: np.real(integrand(y, x, mg, a1, a2, m2_func, q2)),
                0, 1, n=n_points
            )[0]
            integral_imag = fixed_quad(
                lambda y: np.imag(integrand(y, x, mg, a1, a2, m2_func, q2)),
                0, 1, n=n_points
            )[0]
            return integral_real + 1j * integral_imag

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        results.append(integral_value)

    # Retorna escalar se apenas um q_val foi passado
    return np.array(results) if len(results) > 1 else results[0]

In [ ]:
eps_min = 0.0616
mg_min = 0.389
a1_min = 1.50	
a2_min = 2.13

In [ ]:
# Calculates and plot dif sigma 
lst_amp_born_diff = []
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 1e-2
    max_q2   = 0.2001
    q2_step  = 0.001


    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2
        integral_value = full_int(
                mg, a1, a2, mg_model, q2, sqrt_s
        )
        # print(integral_value)
        
        diff_T = integral_value


        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        lst_amp_born_diff.append(amp_value)
        dif_sigma  = differential_sigma(amp_value, s) * scale
        # print(f"q2 = {q2}, diff_t = {integral_value}, amp born = {amp_value:6e} \n")

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    eps_min,
    mg_min,
    a1_min,
    a2_min,
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


In [ ]:


def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


In [ ]:
def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 8 * regge_factor * diff_T  

In [ ]:
import numpy as np
from scipy.special import j0
from scipy.integrate import quad

lst_chi = []
lst_amp_eik = []
lst_diff_sigma = []

eps_rel = 1e-2
eps_abs = 1e-10

q_max_chi = 1.0          # limite de q na Eq. 23
q_max_eik = np.sqrt(0.1) # limite de q (momentum transfer) na Eq. 24
b_max     = 10.0

limit = 500

# ── Eq. 23: χ(s,b) como integral direta em q ─────────────────────────────────
def chi(b_val, mg, a1, a2, eps, m2_func, sqrt_s):

    s_local = sqrt_s ** 2  # [CORREÇÃO 1] s local, não captura variável global

    def integrand(q_val):
        """Integrando complexo — full_int chamado uma única vez por ponto."""
        # [CORREÇÃO 2] integrando único complexo: evita chamar full_int 2x
        q2_val    = q_val ** 2
        t         = -q2_val
        diff_t    = full_int(mg, a1, a2, m2_func, q2_val, sqrt_s)
        born_amp  = amp_calculation(diff_t, s_local, eps, t)
        return (1.0 / s_local) * q_val * j0(b_val * q_val) * born_amp

    # Integrar partes real e imaginária separadamente (quad requer floats)
    real_part, _ = quad(
        lambda q: np.real(integrand(q)),
        0, q_max_chi,
        epsrel=eps_rel, epsabs=eps_abs, limit=limit
    )
    imag_part, _ = quad(
        lambda q: np.imag(integrand(q)),
        0, q_max_chi,
        epsrel=eps_rel, epsabs=eps_abs, limit=limit
    )
    return real_part + 1j * imag_part

In [ ]:
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    m2      = m2_log if model_type == 'log' else m2_pl
    s_local = sqrt_s ** 2  # [CORREÇÃO 3] s local, correto para 7/8/13 TeV

    lst_diff_eik = []

    for q2 in x:
        q_val = np.sqrt(q2)  # [CORREÇÃO 6] nome claro: q_val = |q| [GeV]

        def integrand(b_val):
            """Integrando complexo — chi() chamado uma única vez por ponto."""
            # [CORREÇÃO 5] chi() retorna complexo direto; não duplicar a chamada
            chi_val = chi(b_val, mg, a1, a2, eps, m2, sqrt_s)
            return b_val * j0(q_val * b_val) * (1.0 - np.exp(-chi_val))

        real_part, _ = quad(
            lambda b: np.real(integrand(b)),
            0, b_max,
            epsrel=eps_rel, epsabs=eps_abs, limit=limit
        )
        imag_part, _ = quad(
            lambda b: np.imag(integrand(b)),
            0, b_max,
            epsrel=eps_rel, epsabs=eps_abs, limit=limit
        )

        amp_eik = 1j * s_local * (real_part + 1j * imag_part)

        # [CORREÇÃO 4] |T|² = Re²+Im² — não descartar a parte real
        diff_sigma_eik = (
            (amp_eik.imag**2)
            * (np.pi / s_local**2)
            * 0.389379323          # conversão GeV⁻⁴ → mb
        )
        lst_diff_eik.append(diff_sigma_eik)

    return np.array(lst_diff_eik)

In [ ]:
def model_7(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=7000, model_type='log')

def model_8(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=8000, model_type='log')

def model_13(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=13000, model_type='log')


chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7, verbose=True)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8, verbose=True)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13, verbose=True)


chi2_total = chi2_7 

eps_min = 0.094763
mg_min = 0.93
a1_min = 1.4	
a2_min = 2.69


In [ ]:

minuit_eik = Minuit(
    chi2_total,
    mg = mg_min,
    a1 = a1_min,
    a2 = a2_min,
    eps = eps_min
)


minuit_eik.limits['mg'] = (0.1, 1.0)
minuit_eik.limits['a1'] = (0.5, 3.0)
minuit_eik.limits['a2'] = (0.5, 3.0)
minuit_eik.limits['eps'] = (0.0, 0.1)

minuit_eik.migrad()
minuit_eik.hesse()
